In [ ]:
# Install dependencies
!pip install -q pandas scikit-learn joblib tqdm datasets transformers torch onnx onnxruntime

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
sys.path.insert(0, '.')
print("Repository structure loaded successfully.")

In [ ]:
# Download local assets (GPT-2 Small + Stylometric models)
!python scripts/download_assets.py

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("Loading Toloka Beemo dataset...")
beemo = load_dataset("toloka/beemo")
df = beemo['train'].to_pandas()

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)
train_df = df.loc[train_idx]
test_df = df.loc[test_idx]

def extract_pairs(dataframe):
    records = []
    for _, row in dataframe.iterrows():
        h = row['human_output']
        m = row['model_output']
        if isinstance(h, str) and len(h.strip()) >= 20:
            records.append({'text': h, 'label': 0})
        if isinstance(m, str) and len(m.strip()) >= 20:
            records.append({'text': m, 'label': 1})
    return pd.DataFrame(records)

train_data = extract_pairs(train_df)
test_data = extract_pairs(test_df)
print(f"Train pairs: {len(train_data)}, Test pairs: {len(test_data)}")

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

# Train on a representative balanced subset for speed
train_subset = train_data.sample(n=min(4000, len(train_data)), random_state=42).reset_index(drop=True)
train_dataset = TextDataset(train_subset['text'].tolist(), train_subset['label'].tolist(), tokenizer)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    logging_dir='./logs',
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

print("Training semantic DeBERTa-v3-small on Beemo targets...")
trainer.train()
print("DeBERTa fine-tuning completed.")
model.save_pretrained("deberta_semantic_model")
tokenizer.save_pretrained("deberta_semantic_model")

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

print("Exporting DeBERTa to ONNX...")
model.cpu()
model.eval()

dummy_input = {
    "input_ids": torch.ones(1, 256, dtype=torch.long),
    "attention_mask": torch.ones(1, 256, dtype=torch.long)
}

torch.onnx.export(
    model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    "deberta_semantic_model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14
)

print("Quantizing ONNX model to INT8...")
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(
    "deberta_semantic_model.onnx",
    "deberta_onnx_quantized.onnx",
    weight_type=QuantType.QInt8
)
print("Quantization completed. File size comparison:")
import os
print(f"Original ONNX: {os.path.getsize('deberta_semantic_model.onnx') / 1e6:.2f} MB")
print(f"Quantized ONNX: {os.path.getsize('deberta_onnx_quantized.onnx') / 1e6:.2f} MB")

In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Loading Binoculars models...")
device = "cuda" if torch.cuda.is_available() else "cpu"
observer_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
performer_model = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device)
bino_tokenizer = AutoTokenizer.from_pretrained("gpt2")
bino_tokenizer.pad_token = bino_tokenizer.eos_token

def compute_binoculars(text):
    inputs = bino_tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        obs_logits = observer_model(**inputs).logits
        perf_logits = performer_model(**inputs).logits
        
        shift_labels = inputs["input_ids"][..., 1:].contiguous()
        shift_obs_logits = obs_logits[..., :-1, :].contiguous()
        shift_perf_logits = perf_logits[..., :-1, :].contiguous()
        
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        perf_loss = loss_fct(shift_perf_logits.view(-1, shift_perf_logits.size(-1)), shift_labels.view(-1))
        perf_ppl = torch.exp(perf_loss.mean()).item()
        
        perf_probs = torch.softmax(shift_perf_logits, dim=-1)
        obs_log_probs = torch.log_softmax(shift_obs_logits, dim=-1)
        cross_entropy = -torch.sum(perf_probs * obs_log_probs, dim=-1).mean().item()
        
        score = np.log(perf_ppl) / cross_entropy
        return score

In [ ]:
import joblib
from tool.feature_extractor import extract_features
from tool.neural_detector import compute_perplexity_and_burstiness
from tqdm import tqdm

reg_classifier_path = "models/register_classifier.joblib"
reg_classifier = joblib.load(reg_classifier_path) if os.path.exists(reg_classifier_path) else None

import onnxruntime as ort
ort_sess = ort.InferenceSession("deberta_onnx_quantized.onnx")

def get_deberta_prob(text):
    inputs = tokenizer(text, return_tensors="np", truncation=True, max_length=256, padding=True)
    ort_inputs = {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64)
    }
    logits = ort_sess.run(None, ort_inputs)[0]
    exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    return float(probs[0][1])

def predict_register(feats):
    if reg_classifier is None:
        return "all"
    vec = [feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                             'connector_density', 'hedge_density', 'mean_sent_len',
                             'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']]
    pred = reg_classifier.predict([vec])[0]
    return pred

val_size = min(3000, int(len(test_data) * 0.6))
test_size = min(1500, len(test_data) - val_size)
val_sample, test_sample_eval = train_test_split(test_data, train_size=val_size, test_size=test_size, random_state=42)

records = []
print("Extracting features on validation set...")
for _, row in tqdm(val_sample.iterrows(), total=len(val_sample)):
    text = row['text']
    label = row['label']
    
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    
    stylo_vector = [feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                      'connector_density', 'hedge_density', 'mean_sent_len',
                                      'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']]
    try:
        neural = compute_perplexity_and_burstiness(text)
        ppl = neural['perplexity']
        burst = neural['burstiness']
    except:
        ppl, burst = 50.0, 1.0
        
    try:
        bino = compute_binoculars(text)
    except:
        bino = 0.95
        
    deberta_prob = get_deberta_prob(text)
    reg = predict_register(feats)
    
    records.append({
        'vector': stylo_vector + [ppl, burst, bino, deberta_prob],
        'label': label,
        'register': reg
    })

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

ensemblers = {}
X_all = np.array([r['vector'] for r in records])
y_all = np.array([r['label'] for r in records])

ensembler_all = LogisticRegression(max_iter=1000)
ensembler_all.fit(X_all, y_all)
ensemblers['all'] = ensembler_all
print(f"Overall Train AUC: {roc_auc_score(y_all, ensembler_all.predict_proba(X_all)[:, 1]):.4f}")

df_val = pd.DataFrame(records)
for reg in ['news', 'academic', 'social', 'creative']:
    reg_records = df_val[df_val['register'] == reg]
    if len(reg_records) >= 50:
        X_reg = np.array(reg_records['vector'].tolist())
        y_reg = np.array(reg_records['label'].tolist())
        
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_reg, y_reg)
        ensemblers[reg] = clf
        print(f"Register '{reg}' Train AUC: {roc_auc_score(y_reg, clf.predict_proba(X_reg)[:, 1]):.4f}")
    else:
        ensemblers[reg] = ensembler_all
        print(f"Register '{reg}' fallback to overall model (not enough examples: {len(reg_records)})")

joblib.dump(ensemblers, "beemo_register_ensemblers.joblib")

In [ ]:
X_test_all = []
y_test_all = []
y_test_preds = []

print("Evaluating Register-Aware Ensembler on held-out test cases...")
for _, row in tqdm(test_sample_eval.iterrows(), total=len(test_sample_eval)):
    text = row['text']
    label = row['label']
    
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    
    stylo_vector = [feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                      'connector_density', 'hedge_density', 'mean_sent_len',
                                      'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']]
    try:
        neural = compute_perplexity_and_burstiness(text)
        ppl = neural['perplexity']
        burst = neural['burstiness']
    except:
        ppl, burst = 50.0, 1.0
        
    try:
        bino = compute_binoculars(text)
    except:
        bino = 0.95
        
    deberta_prob = get_deberta_prob(text)
    reg = predict_register(feats)
    
    vec = stylo_vector + [ppl, burst, bino, deberta_prob]
    clf = ensemblers.get(reg, ensemblers['all'])
    pred_prob = clf.predict_proba([vec])[0][1]
    
    X_test_all.append(vec)
    y_test_all.append(label)
    y_test_preds.append(pred_prob)

test_auc = roc_auc_score(y_test_all, y_test_preds)
print(f"\n--- FINAL SOTA PIPELINE TEST AUC: {test_auc:.4f} ---")

In [ ]:
import shutil
shutil.copy("deberta_onnx_quantized.onnx", "/kaggle/working/deberta_onnx_quantized.onnx")
shutil.copy("beemo_register_ensemblers.joblib", "/kaggle/working/beemo_register_ensemblers.joblib")
print("All SOTA artifacts copied to /kaggle/working/ and ready for download.")